# Heart Disease UCI - EDA and model training

Runs the same code as the `heartml` package so notebook results match the CLI and CI.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))
import pandas as pd
from IPython.display import Image, display
from heartml import config, data, eda

## 1. Acquire and clean

In [ ]:
data.download_raw()
raw = data.load_raw()
df = data.clean(raw)
df.to_csv(config.CLEAN_FILE, index=False)
print(raw.shape, '->', df.shape)
df.head()

In [ ]:
data.missing_report(df)

In [ ]:
df.describe().T.round(2)

## 2. Exploratory analysis

In [ ]:
eda.main()
for f in ['class_balance', 'missing_values', 'numeric_histograms', 'numeric_boxplots', 'categorical_counts', 'correlation_heatmap']:
    display(Image(filename=str(config.FIGURES_DIR / f'{f}.png')))

In [ ]:
pd.Series(eda.summary(df)['correlation_with_target']).sort_values().plot.barh(figsize=(6, 5), title='Correlation with target');

## 3. Train, tune and track (MLflow)

In [ ]:
from heartml import train
import mlflow
mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
mlflow.set_experiment(config.MLFLOW_EXPERIMENT)
X, y = train.load_dataset()
X_train, X_test, y_train, y_test = train.split(X, y)
results = [train.train_one(m, X_train, X_test, y_train, y_test) for m in train.CANDIDATES]
pd.DataFrame([{'model': r['name'], **r['metrics']} for r in results]).round(3)

In [ ]:
best = max(results, key=lambda r: r['metrics']['cv_roc_auc_mean'])
meta = train.save_best(best, X_train)
meta

## 4. Inference

In [ ]:
from heartml.predict import HeartModel
model = HeartModel()
model.predict([json.load(open('sample_input.json'))])